# Modeling using random forest

In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pyproj import Transformer
from sklearn.impute import SimpleImputer

In [2]:
import psutil

# Total and available RAM
mem = psutil.virtual_memory()

print(f"Total RAM:     {mem.total / (1024**3):.2f} GB")
print(f"Available RAM: {mem.available / (1024**3):.2f} GB")
print(f"RAM in use:    {mem.percent}%")

Total RAM:     15.71 GB
Available RAM: 8.01 GB
RAM in use:    49.0%


In [3]:
import os

file_path = 'CIMIS_WBGT_all_2000_2025_integrated_new.csv'

file_size_gb = os.path.getsize(file_path) / (1024**3)

print(f"CSV file size: {file_size_gb:.2f} GB")

CSV file size: 9.77 GB


In [4]:
# load dataset --- just 500,000 rows for now to test
integrated = pd.read_csv('CIMIS_WBGT_all_2000_2025_integrated_new.csv', nrows=2000000)

In [5]:
integrated.columns

Index(['station_id', 'year', 'month', 'day', 'hour', 'dday', 'lat', 'lon',
       'GEOID', 'date_local', 'datetime_utc', 'year_utc', 'month_utc',
       'day_utc', 'hour_utc', 'dday_utc', 'solar', 'Tair', 'relhum', 'speed',
       'elev_m', 'cza', 'fdir', 'pres', 'zspeed', 'dT', 'urban', 'wbgt',
       'aster_elev_mean', 'aster_elev_sd', 'aster_elev_min', 'aster_elev_max',
       'aster_elev_range', 'aster_elev_median', 'aster_elev_q25',
       'aster_elev_q75', 'aster_elev_iqr', 'nlcd_urban', 'nlcd_imp_frac',
       'nlcd_tree_frac', 'nlcd_LC11_frac', 'nlcd_LC21_frac', 'nlcd_LC22_frac',
       'nlcd_LC23_frac', 'nlcd_LC24_frac', 'nlcd_LC31_frac', 'nlcd_LC41_frac',
       'nlcd_LC42_frac', 'nlcd_LC43_frac', 'nlcd_LC52_frac', 'nlcd_LC71_frac',
       'nlcd_LC81_frac', 'nlcd_LC82_frac', 'nlcd_LC90_frac', 'nlcd_LC95_frac',
       'nlcd_developed_frac', 'nlcd_forest_frac', 'nlcd_shrub_frac',
       'nlcd_water_frac', 'nlcd_crop_frac', 'nlcd_wetland_frac', 'modis_ndvi',
       'gridmet_rmax

In [6]:
with pd.option_context('display.max_columns', None): display(integrated.head())

,station_id,year,month,day,hour,dday,lat,lon,GEOID,date_local,datetime_utc,year_utc,month_utc,day_utc,hour_utc,dday_utc,solar,Tair,relhum,speed,elev_m,cza,fdir,pres,zspeed,dT,urban,wbgt,aster_elev_mean,aster_elev_sd,aster_elev_min,aster_elev_max,aster_elev_range,aster_elev_median,aster_elev_q25,aster_elev_q75,aster_elev_iqr,nlcd_urban,nlcd_imp_frac,nlcd_tree_frac,nlcd_LC11_frac,nlcd_LC21_frac,nlcd_LC22_frac,nlcd_LC23_frac,nlcd_LC24_frac,nlcd_LC31_frac,nlcd_LC41_frac,nlcd_LC42_frac,nlcd_LC43_frac,nlcd_LC52_frac,nlcd_LC71_frac,nlcd_LC81_frac,nlcd_LC82_frac,nlcd_LC90_frac,nlcd_LC95_frac,nlcd_developed_frac,nlcd_forest_frac,nlcd_shrub_frac,nlcd_water_frac,nlcd_crop_frac,nlcd_wetland_frac,modis_ndvi,gridmet_rmax,gridmet_rmin,gridmet_sph,gridmet_srad,gridmet_tmmn,gridmet_tmmx,gridmet_vs,daymet_vp,daymet_tmin,daymet_tmax,daymet_srad,era5_wbgt
0,5,2000,1,1,0,1.000000,35.532556,-119.28179,144462,2000-01-01,2000-01-01T08:00:00Z,2000,1,1,8,1.333333,0,4.6,87,1.4,109.728,0.0,0.0,1000.137467,10,-0.052,0,3.177265,106.555916,2.755774,93,116,23,107,105,108,3,0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,4.747215
1,5,2000,1,1,1,1.041667,35.532556,-119.28179,144462,2000-01-01,2000-01-01T09:00:00Z,2000,1,1,9,1.375000,1,3.8,88,1.1,109.728,0.0,0.0,1000.137467,10,-0.052,0,2.358547,106.555916,2.755774,93,116,23,107,105,108,3,0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,4.666534
2,5,2000,1,1,2,1.083333,35.532556,-119.28179,144462,2000-01-01,2000-01-01T10:00:00Z,2000,1,1,10,1.416667,1,2.6,91,1.3,109.728,0.0,0.0,1000.137467,10,-0.052,0,1.361490,106.555916,2.755774,93,116,23,107,105,108,3,0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,4.328669
3,5,2000,1,1,3,1.125000,35.532556,-119.28179,144462,2000-01-01,2000-01-01T11:00:00Z,2000,1,1,11,1.458333,1,1.8,92,1.2,109.728,0.0,0.0,1000.137467,10,-0.052,0,0.559265,106.555916,2.755774,93,116,23,107,105,108,3,0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,3.766729
4,5,2000,1,1,4,1.166667,35.532556,-119.28179,144462,2000-01-01,2000-01-01T12:00:00Z,2000,1,1,12,1.500000,1,1.2,94,0.8,109.728,0.0,0.0,1000.137467,10,-0.052,0,-0.097021,106.555916,2.755774,93,116,23,107,105,108,3,0,0.0035,0.0001,0.0,0.0045,0.0053,0.0009,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.9893,0.0,0.0,0.0107,0.0,0.0,0.0,0.9893,0.0,NaN,83.257904,29.14911,0.003764,114.133125,274.34543,288.944,2.768005,339.70248,2.859743,14.75241,247.11838,3.504375


## preprocessing test dataset (month = Jan 2003)

In [7]:
# subset to January 2003 as the test month
df = integrated[
    (integrated["year"] == 2003) &
    (integrated["month"] == 1)
].copy()

print("Number of rows:", len(df))
print("Unique stations:", df["station_id"].nunique())
df.head()

Number of rows: 19252
Unique stations: 26


,station_id,year,month,day,hour,dday,lat,lon,GEOID,date_local,...,gridmet_sph,gridmet_srad,gridmet_tmmn,gridmet_tmmx,gridmet_vs,daymet_vp,daymet_tmin,daymet_tmax,daymet_srad,era5_wbgt
1778272,2,2003,1,1,0,1.000000,36.336222,-120.11291,191939,2003-01-01,...,0.004867,83.20817,274.904,280.81198,1.628174,663.1033,2.058992,13.996434,258.17215,4.909988
1778273,2,2003,1,1,1,1.041667,36.336222,-120.11291,191939,2003-01-01,...,0.004867,83.20817,274.904,280.81198,1.628174,663.1033,2.058992,13.996434,258.17215,3.500152
1778274,2,2003,1,1,2,1.083333,36.336222,-120.11291,191939,2003-01-01,...,0.004867,83.20817,274.904,280.81198,1.628174,663.1033,2.058992,13.996434,258.17215,3.404094
1778275,2,2003,1,1,3,1.125000,36.336222,-120.11291,191939,2003-01-01,...,0.004867,83.20817,274.904,280.81198,1.628174,663.1033,2.058992,13.996434,258.17215,3.158316
1778276,2,2003,1,1,4,1.166667,36.336222,-120.11291,191939,2003-01-01,...,0.004867,83.20817,274.904,280.81198,1.628174,663.1033,2.058992,13.996434,258.17215,2.076860


In [8]:
# Missingness within the selected month
print(
    df[["wbgt", "era5_wbgt", "modis_ndvi"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

# Station coverage
print(
    df.groupby("station_id")
    .size()
    .describe()
)

wbgt          0.000052
era5_wbgt     0.000000
modis_ndvi    0.000000
dtype: float64
count     26.000000
mean     740.461538
std        8.095583
min      710.000000
25%      743.000000
50%      744.000000
75%      744.000000
max      744.000000
dtype: float64


In [9]:
features = [
    "era5_wbgt",

    # Hourly meteorology
    "solar", "Tair", "relhum", "speed",
    "elev_m", "cza", "fdir", "pres",
    "zspeed", "dT",
    
    "urban",

    # ASTER terrain statistics
    "aster_elev_mean", "aster_elev_sd",
    "aster_elev_min", "aster_elev_max",
    "aster_elev_range", "aster_elev_median",
    "aster_elev_q25", "aster_elev_q75",
    "aster_elev_iqr",

    # NLCD land-cover variables
    "nlcd_urban", "nlcd_imp_frac", "nlcd_tree_frac",
    "nlcd_LC11_frac", "nlcd_LC21_frac",
    "nlcd_LC22_frac", "nlcd_LC23_frac",
    "nlcd_LC24_frac", "nlcd_LC31_frac",
    "nlcd_LC41_frac", "nlcd_LC42_frac",
    "nlcd_LC43_frac", "nlcd_LC52_frac",
    "nlcd_LC71_frac", "nlcd_LC81_frac",
    "nlcd_LC82_frac", "nlcd_LC90_frac",
    "nlcd_LC95_frac", "nlcd_developed_frac",
    "nlcd_forest_frac", "nlcd_shrub_frac",
    "nlcd_water_frac", "nlcd_crop_frac",
    "nlcd_wetland_frac",

    # Vegetation
    "modis_ndvi",

    # GridMET
    "gridmet_rmax", "gridmet_rmin", "gridmet_sph",
    "gridmet_srad", "gridmet_tmmn", "gridmet_tmmx",
    "gridmet_vs",

    # Daymet
    "daymet_vp", "daymet_tmin", "daymet_tmax",
    "daymet_srad",
]

In [10]:
# Unique station locations
stations = df[
    ["station_id", "lat", "lon"]
].drop_duplicates("station_id")

print("Unique stations:", len(stations))
print("\nLatitude range:")
print(stations["lat"].agg(["min", "max"]))

print("\nLongitude range:")
print(stations["lon"].agg(["min", "max"]))

# Observations per station
coverage = df.groupby("station_id").size()

print("\nStations with fewer than 744 hours:",
      (coverage < 744).sum())

# Missingness of all predictors
print("\nMost incomplete predictors:")
print(
    df[features]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

Unique stations: 26

Latitude range:
min    32.806183
max    41.958869
Name: lat, dtype: float64

Longitude range:
min   -122.79993
max   -115.41585
Name: lon, dtype: float64

Stations with fewer than 744 hours: 8

Most incomplete predictors:
era5_wbgt    0.0
solar        0.0
Tair         0.0
relhum       0.0
speed        0.0
elev_m       0.0
cza          0.0
fdir         0.0
pres         0.0
zspeed       0.0
dtype: float64


In [11]:
# --------------------------------------------------
# 1. remove rows missing target variable
# --------------------------------------------------

df = df.dropna(subset=["wbgt"]).copy()

# ============================================================
# 2. check predictors
# ============================================================

# Check whether every expected predictor exists.
missing_cols = [c for c in features if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing predictor columns: {missing_cols}")

# Convert predictors to numeric values.
df[features] = df[features].apply(
    pd.to_numeric, errors="coerce"
)

# ============================================================
# 3. Project longitude/latitude to metric coordinates
# ============================================================

# EPSG:3310 is California Albers; units are meters.
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3310", always_xy=True)

df["x"], df["y"] = transformer.transform(
    df["lon"].to_numpy(),
    df["lat"].to_numpy(),
)

coords = df[["x", "y"]].to_numpy()

# ============================================================
# 4. Create train/test groups by individual stations
# ============================================================

# Check that station IDs are available.
if df["station_id"].isna().any():
    raise ValueError("Some rows have missing station_id values.")

# Each unique station_id represents one site.
groups = df["station_id"]

In [12]:
# ============================================================
# 5. Split into training and test spatial blocks
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["station_id"])
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

# ============================================================
# 6. Impute missing predictor values
# ============================================================

# Fit the imputer on training data only.
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(imputer.fit_transform(train[features]), columns=features, index=train.index)
X_test = pd.DataFrame(imputer.transform(test[features]), columns=features, index=test.index)

y_train = train["wbgt"].to_numpy()
y_test = test["wbgt"].to_numpy()

coords_train = train[["x", "y"]].to_numpy()
coords_test = test[["x", "y"]].to_numpy()

In [13]:
train_stations = set(train["station_id"].unique())
test_stations = set(test["station_id"].unique())

print("Training observations:", len(train))
print("Testing observations:", len(test))

print("Training stations:", len(train_stations))
print("Testing stations:", len(test_stations))

print("Overlapping stations:", train_stations.intersection(test_stations))

Training observations: 14836
Testing observations: 4415
Training stations: 20
Testing stations: 6
Overlapping stations: set()


## check correlation of wbgt with predictors

In [14]:
len(features)

57

In [15]:
# Correlations with WBGT in the training sample
corr = (
    train[features + ["wbgt"]]
    .corr(numeric_only=True)["wbgt"]
    .drop("wbgt")
    .sort_values(key=abs, ascending=False)
)

print(corr)

Tair                   0.928892
era5_wbgt              0.813023
solar                  0.684497
cza                    0.662718
daymet_tmin            0.544871
fdir                   0.518323
gridmet_tmmn           0.504804
relhum                -0.453001
daymet_tmax            0.383843
aster_elev_min        -0.375235
aster_elev_q25        -0.371286
aster_elev_median     -0.368728
aster_elev_mean       -0.368566
elev_m                -0.366199
aster_elev_q75        -0.366125
pres                   0.363489
aster_elev_max        -0.358333
gridmet_tmmx           0.341965
daymet_vp              0.313873
gridmet_sph            0.301685
modis_ndvi             0.251860
nlcd_LC81_frac        -0.245654
nlcd_LC24_frac        -0.237083
speed                  0.218480
nlcd_LC90_frac        -0.217757
gridmet_srad           0.168617
nlcd_LC21_frac         0.155889
aster_elev_range       0.110304
daymet_srad            0.104830
gridmet_rmax          -0.100486
aster_elev_iqr         0.098373
aster_el

In [21]:
?RandomForestRegressor

Init signature:
RandomForestRegressor(
    n_estimators=100,
    *,
    criterion='squared_error',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=1.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=False,
    n_jobs=None,
    random_state=None,
    verbose=0,
    warm_start=False,
    ccp_alpha=0.0,
    max_samples=None,
    monotonic_cst=None,
)
Docstring:     
A random forest regressor.

A random forest is a meta estimator that fits a number of decision tree
regressors on various sub-samples of the dataset and uses averaging to
improve the predictive accuracy and control over-fitting.
Trees in the forest use the best split strategy, i.e. equivalent to passing
`splitter="best"` to the underlying :class:`~sklearn.tree.DecisionTreeRegressor`.
The sub-sample size is controlled with the `max_samples` parameter if
`bootstrap=True` (default), otherwise the whole dataset is use

## 10-fold cross validation

For the WBGT Random Forest models, use 10-fold cross-validation (CV) to evaluate how consistently each model performs across different data splits.

In [20]:
# Model 1: All predictors, excluding the baseline WBGT
features_all = [
    col for col in features
    if col != baseline
]

# Model 2: Selected 17 predictors, excluding ERA5 WBGT
features_selected = [
    "Tair",
    "solar",
    "cza",
    "daymet_tmin",
    "gridmet_tmmn",
    "fdir",
    "relhum",
    "aster_elev_min",
    "aster_elev_q25",
    "aster_elev_mean",
    "aster_elev_median",
    "daymet_tmax",
    "aster_elev_q75",
    "elev_m",
    "aster_elev_max",
    "pres",
    "gridmet_tmmx",
]

# Model 3: Top 6 predictors, excluding ERA5 WBGT
features_top6 = [
    "Tair",
    "solar",
    "cza",
    "daymet_tmin",
    "gridmet_tmmn",
    "fdir",
]

In [21]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ============================================================
# 1. Define the target and three models
# ============================================================

target = "wbgt"                 # Observed CIMIS WBGT
baseline = "era5_wbgt"          # ERA5 WBGT baseline
residual_target = "wbgt_residual"

models = {
    "RF - All predictors": features_all,
    "RF - Selected predictors": features_selected,
    "RF - Top predictors": features_top6,
}

# Ensure ERA5 WBGT and observed WBGT are not predictors
for model_name, feature_cols in models.items():
    if target in feature_cols:
        raise ValueError(
            f"{model_name} includes observed WBGT as a predictor."
        )
    if baseline in feature_cols:
        raise ValueError(
            f"{model_name} includes era5_wbgt as a predictor."
        )

# Check predictor columns exist
for model_name, feature_cols in models.items():
    missing = set(feature_cols) - set(train.columns)
    if missing:
        raise ValueError(
            f"{model_name}: missing columns {missing}"
        )

# ============================================================
# 2. Prepare training data
# ============================================================

# Require observed CIMIS WBGT, ERA5 WBGT, and station ID
cv_data = train.dropna(
    subset=[target, baseline, "station_id"]
).copy()

# Residual = CIMIS WBGT - ERA5 WBGT
cv_data[residual_target] = (
    cv_data[target] - cv_data[baseline]
)

# The model learns to predict the residual
y = cv_data[residual_target].to_numpy()

# Preserve the ERA5 baseline for reconstructing WBGT
era5_values = cv_data[baseline].to_numpy()

# Station IDs determine the cross-validation groups
groups = cv_data["station_id"].to_numpy()

n_stations = cv_data["station_id"].nunique()

print("Training observations:", len(cv_data))
print("Training stations:", n_stations)
print("Training target: CIMIS WBGT - ERA5 WBGT")

if n_stations < 10:
    raise ValueError(
        f"Only {n_stations} stations available. "
        "10-fold station-based CV requires at least 10 stations."
    )

# ============================================================
# 3. Set up 10-fold station-based cross-validation
# ============================================================

cv = GroupKFold(n_splits=10)

# ============================================================
# 4. Run cross-validation for all three models
# ============================================================

cv_results = []
start_all = time.time()

for model_name, feature_cols in models.items():

    print(f"\nRunning: {model_name}")

    X = cv_data[feature_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X, y, groups=groups), start=1
    ):

        # Split observations by station
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y[train_idx]  # Training residuals
        y_val = y[val_idx]      # Validation residuals

        # ERA5 baselines for validation observations
        era5_val = era5_values[val_idx]

        # Actual CIMIS WBGT for validation observations
        wbgt_val = cv_data[target].to_numpy()[val_idx]

        # Confirm stations do not overlap
        train_stations = set(groups[train_idx])
        val_stations = set(groups[val_idx])

        assert train_stations.isdisjoint(val_stations)

        # Fit imputer on this fold's training data only
        imputer = SimpleImputer(strategy="median")

        X_train_imp = imputer.fit_transform(X_train)
        X_val_imp = imputer.transform(X_val)

        # Train Random Forest to predict residuals
        rf = RandomForestRegressor(
            n_estimators=300,
            max_features=1.0,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
        )

        rf.fit(X_train_imp, y_train)

        # Predict residuals for validation observations
        predicted_residual = rf.predict(X_val_imp)

        # Reconstruct predicted CIMIS WBGT
        predicted_wbgt = era5_val + predicted_residual

        # ----------------------------------------------------
        # Primary metrics: reconstructed CIMIS WBGT
        # ----------------------------------------------------

        rmse = np.sqrt(
            mean_squared_error(wbgt_val, predicted_wbgt)
        )
        mae = mean_absolute_error(
            wbgt_val, predicted_wbgt
        )
        bias = np.mean(predicted_wbgt - wbgt_val)
        r2 = r2_score(wbgt_val, predicted_wbgt)

        # ----------------------------------------------------
        # Additional metrics: residual predictions
        # ----------------------------------------------------

        residual_rmse = np.sqrt(
            mean_squared_error(y_val, predicted_residual)
        )
        residual_mae = mean_absolute_error(
            y_val, predicted_residual
        )
        residual_bias = np.mean(
            predicted_residual - y_val
        )
        residual_r2 = r2_score(y_val, predicted_residual)

        cv_results.append({
            "Model": model_name,
            "Fold": fold,
            "Train_rows": len(train_idx),
            "Validation_rows": len(val_idx),
            "Validation_stations": len(val_stations),

            # Metrics for reconstructed CIMIS WBGT
            "WBGT_RMSE": rmse,
            "WBGT_MAE": mae,
            "WBGT_Bias": bias,
            "WBGT_R2": r2,

            # Metrics for residual predictions
            "Residual_RMSE": residual_rmse,
            "Residual_MAE": residual_mae,
            "Residual_Bias": residual_bias,
            "Residual_R2": residual_r2,
        })

        print(
            f"Fold {fold:02d}/10 | "
            f"WBGT RMSE={rmse:.4f} | "
            f"WBGT MAE={mae:.4f} | "
            f"WBGT R2={r2:.4f} | "
            f"Residual RMSE={residual_rmse:.4f}"
        )

print(
    f"\nTotal cross-validation time: "
    f"{(time.time() - start_all) / 60:.2f} minutes"
)

# ============================================================
# 5. Display results for each fold
# ============================================================

cv_results_df = pd.DataFrame(cv_results)

print("\nResults for each fold:")
print(cv_results_df.round(4).to_string(index=False))

# ============================================================
# 6. Summarize mean and standard deviation across folds
# ============================================================

wbgt_metrics = [
    "WBGT_RMSE",
    "WBGT_MAE",
    "WBGT_Bias",
    "WBGT_R2",
]

residual_metrics = [
    "Residual_RMSE",
    "Residual_MAE",
    "Residual_Bias",
    "Residual_R2",
]

# Primary summary: reconstructed CIMIS WBGT
wbgt_summary = (
    cv_results_df
    .groupby("Model")[wbgt_metrics]
    .agg(["mean", "std"])
)

print("\n10-fold CV summary: reconstructed CIMIS WBGT")
print(wbgt_summary.round(4))

# Additional summary: residual prediction performance
residual_summary = (
    cv_results_df
    .groupby("Model")[residual_metrics]
    .agg(["mean", "std"])
)

print("\n10-fold CV summary: residual predictions")
print(residual_summary.round(4))

# ============================================================
# 7. Save results
# ============================================================

cv_results_df.to_csv(
    "wbgt_rf_residual_10fold_cv_each_fold.csv",
    index=False,
)

wbgt_summary.to_csv(
    "wbgt_rf_residual_10fold_cv_wbgt_summary.csv"
)

residual_summary.to_csv(
    "wbgt_rf_residual_10fold_cv_residual_summary.csv"
)

print("\nSaved:")
print("- wbgt_rf_residual_10fold_cv_each_fold.csv")
print("- wbgt_rf_residual_10fold_cv_wbgt_summary.csv")
print("- wbgt_rf_residual_10fold_cv_residual_summary.csv")

Training observations: 14836
Training stations: 20
Training target: CIMIS WBGT - ERA5 WBGT

Running: RF - All predictors
Fold 01/10 | WBGT RMSE=2.7213 | WBGT MAE=2.0874 | WBGT R2=0.7738 | Residual RMSE=2.7213
Fold 02/10 | WBGT RMSE=2.2715 | WBGT MAE=1.8098 | WBGT R2=0.7763 | Residual RMSE=2.2715
Fold 03/10 | WBGT RMSE=2.4948 | WBGT MAE=1.9942 | WBGT R2=0.7650 | Residual RMSE=2.4948
Fold 04/10 | WBGT RMSE=2.2743 | WBGT MAE=1.7103 | WBGT R2=0.6953 | Residual RMSE=2.2743
Fold 05/10 | WBGT RMSE=2.0222 | WBGT MAE=1.6284 | WBGT R2=0.7569 | Residual RMSE=2.0222
Fold 06/10 | WBGT RMSE=1.9894 | WBGT MAE=1.5417 | WBGT R2=0.7238 | Residual RMSE=1.9894
Fold 07/10 | WBGT RMSE=1.8747 | WBGT MAE=1.4621 | WBGT R2=0.8102 | Residual RMSE=1.8747
Fold 08/10 | WBGT RMSE=2.1090 | WBGT MAE=1.7115 | WBGT R2=0.8214 | Residual RMSE=2.1090
Fold 09/10 | WBGT RMSE=1.6817 | WBGT MAE=1.3108 | WBGT R2=0.8703 | Residual RMSE=1.6817
Fold 10/10 | WBGT RMSE=1.8575 | WBGT MAE=1.4418 | WBGT R2=0.7993 | Residual RMSE=1.8575

## randomized hyperparameter search (using 5-fold cv)

In [22]:

from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# --------------------------------------------------
# 1. Prepare residual prediction data
# --------------------------------------------------

cv_data = train.dropna(
    subset=["wbgt", "era5_wbgt", "station_id"]
).copy()

cv_data["wbgt_residual"] = (
    cv_data["wbgt"] - cv_data["era5_wbgt"]
)

y = cv_data["wbgt_residual"].to_numpy()
groups = cv_data["station_id"].to_numpy()

feature_cols = features_all

# Ensure target and baseline are not predictors
if "wbgt" in feature_cols or "era5_wbgt" in feature_cols:
    raise ValueError("Target or baseline included in predictors.")

X = (
    cv_data[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
)

# --------------------------------------------------
# 2. Pipeline
# --------------------------------------------------

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

# --------------------------------------------------
# 3. Smaller hyperparameter search space
# --------------------------------------------------

param_grid = {
    "rf__n_estimators": [100],
    "rf__max_features": [0.5, 0.75, 1.0],
    "rf__min_samples_leaf": [2, 4, 8],
    "rf__max_depth": [None, 15, 25],
    "rf__min_samples_split": [2, 5, 10],
    "rf__max_samples": [0.7, 1.0],
}

# --------------------------------------------------
# 4. Five-fold station-based CV
# --------------------------------------------------

cv = GroupKFold(n_splits=5)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=12,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=2,
    refit=False
)

search.fit(X, y, groups=groups)

# --------------------------------------------------
# 5. Results
# --------------------------------------------------

print("Best parameters:")
print(search.best_params_)

print("\nBest mean CV residual RMSE:")
print(-search.best_score_)

results = pd.DataFrame(search.cv_results_)

results["RMSE"] = -results["mean_test_score"]
results["RMSE_SD"] = results["std_test_score"]

results = results.sort_values("RMSE")

print("\nTop parameter combinations:")
print(
    results[
        ["RMSE", "RMSE_SD", "params"]
    ].head(10).to_string(index=False)
)


Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END rf__max_depth=25, rf__max_features=1.0, rf__max_samples=1.0, rf__min_samples_leaf=4, rf__min_samples_split=10, rf__n_estimators=100; total time=   6.6s
[CV] END rf__max_depth=25, rf__max_features=1.0, rf__max_samples=1.0, rf__min_samples_leaf=4, rf__min_samples_split=10, rf__n_estimators=100; total time=   6.8s
[CV] END rf__max_depth=25, rf__max_features=1.0, rf__max_samples=1.0, rf__min_samples_leaf=4, rf__min_samples_split=10, rf__n_estimators=100; total time=   6.6s
[CV] END rf__max_depth=25, rf__max_features=1.0, rf__max_samples=1.0, rf__min_samples_leaf=4, rf__min_samples_split=10, rf__n_estimators=100; total time=   6.8s
[CV] END rf__max_depth=25, rf__max_features=1.0, rf__max_samples=1.0, rf__min_samples_leaf=4, rf__min_samples_split=10, rf__n_estimators=100; total time=   6.7s
[CV] END rf__max_depth=25, rf__max_features=0.5, rf__max_samples=0.7, rf__min_samples_leaf=2, rf__min_samples_split=5, rf__n_estimator

In [24]:
# top 5 parameter combinations

for i, row in results.head(5).iterrows():

    print(f"\nRank {i + 1}")
    print(f"Mean CV RMSE: {row['RMSE']:.4f} °C")
    print(f"SD across folds: {row['RMSE_SD']:.4f} °C")

    for parameter, value in row["params"].items():
        print(f"  {parameter}: {value}")


Rank 1
Mean CV RMSE: 2.2016 °C
SD across folds: 0.1934 °C
  rf__n_estimators: 100
  rf__min_samples_split: 5
  rf__min_samples_leaf: 2
  rf__max_samples: 0.7
  rf__max_features: 0.5
  rf__max_depth: 25

Rank 2
Mean CV RMSE: 2.2073 °C
SD across folds: 0.2223 °C
  rf__n_estimators: 100
  rf__min_samples_split: 2
  rf__min_samples_leaf: 4
  rf__max_samples: 1.0
  rf__max_features: 0.75
  rf__max_depth: 15

Rank 3
Mean CV RMSE: 2.2114 °C
SD across folds: 0.2620 °C
  rf__n_estimators: 100
  rf__min_samples_split: 5
  rf__min_samples_leaf: 2
  rf__max_samples: 0.7
  rf__max_features: 0.75
  rf__max_depth: None

Rank 4
Mean CV RMSE: 2.2156 °C
SD across folds: 0.2380 °C
  rf__n_estimators: 100
  rf__min_samples_split: 10
  rf__min_samples_leaf: 2
  rf__max_samples: 1.0
  rf__max_features: 0.75
  rf__max_depth: None

Rank 5
Mean CV RMSE: 2.2167 °C
SD across folds: 0.1846 °C
  rf__n_estimators: 100
  rf__min_samples_split: 5
  rf__min_samples_leaf: 2
  rf__max_samples: 0.7
  rf__max_features: 0

## one year modeling - preprocessing

In [36]:
features = [
    "era5_wbgt",

    # Hourly meteorology
    "solar", "Tair", "relhum", "speed",
    "elev_m", "cza", "fdir", "pres",
    "zspeed", "dT",
    
    "urban",

    # ASTER terrain statistics
    "aster_elev_mean", "aster_elev_sd",
    "aster_elev_min", "aster_elev_max",
    "aster_elev_range", "aster_elev_median",
    "aster_elev_q25", "aster_elev_q75",
    "aster_elev_iqr",

    # NLCD land-cover variables
    "nlcd_urban", "nlcd_imp_frac", "nlcd_tree_frac",
    "nlcd_LC11_frac", "nlcd_LC21_frac",
    "nlcd_LC22_frac", "nlcd_LC23_frac",
    "nlcd_LC24_frac", "nlcd_LC31_frac",
    "nlcd_LC41_frac", "nlcd_LC42_frac",
    "nlcd_LC43_frac", "nlcd_LC52_frac",
    "nlcd_LC71_frac", "nlcd_LC81_frac",
    "nlcd_LC82_frac", "nlcd_LC90_frac",
    "nlcd_LC95_frac", "nlcd_developed_frac",
    "nlcd_forest_frac", "nlcd_shrub_frac",
    "nlcd_water_frac", "nlcd_crop_frac",
    "nlcd_wetland_frac",

    # Vegetation
    "modis_ndvi",

    # GridMET
    "gridmet_rmax", "gridmet_rmin", "gridmet_sph",
    "gridmet_srad", "gridmet_tmmn", "gridmet_tmmx",
    "gridmet_vs",

    # Daymet
    "daymet_vp", "daymet_tmin", "daymet_tmax",
    "daymet_srad",
]

In [37]:
# ============================================================
# 1. Load all observations from 2003
# ============================================================

file_path = "CIMIS_WBGT_all_2000_2025_integrated_new.csv"

# Read only columns needed for modeling and evaluation.
required_cols = list(dict.fromkeys(
    features + [
        "station_id",
        "year",
        "month",
        "date_local",
        "datetime_utc",
        "lat",
        "lon",
        "wbgt"
    ]
))

chunks_2003 = []

for chunk in pd.read_csv(
    file_path,
    usecols=required_cols,
    chunksize=500_000,
    dtype={"station_id": "string"}
):
    # Retain all rows from 2003
    chunk_2003 = chunk.loc[
        pd.to_numeric(chunk["year"], errors="coerce") == 2003
    ]

    if not chunk_2003.empty:
        chunks_2003.append(chunk_2003)

# Combine the 2003 observations
df_year = pd.concat(
    chunks_2003,
    ignore_index=True
)

print("Year:", df_year["year"].unique())
print("Number of observations:", len(df_year))
print("Number of stations:", df_year["station_id"].nunique())
print("Months represented:", sorted(df_year["month"].unique()))

# Check station coverage
print("\nObservations per station:")
print(df_year.groupby("station_id").size().describe())


Year: [2003]
Number of observations: 573485
Number of stations: 66
Months represented: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

Observations per station:
count      66.000000
mean     8689.166667
std       249.286279
min      6802.000000
25%      8719.000000
50%      8753.000000
75%      8758.750000
max      8760.000000
dtype: float64


In [38]:
baseline = "era5_wbgt"
target = "wbgt"

features_all = [
    col for col in features
    if col != baseline
]

In [39]:
# ============================================================
# 2. Examine annual data coverage and missingness
# ============================================================

print("\nMissingness in important variables:")
print(
    df_year[
        ["wbgt", "era5_wbgt", "modis_ndvi"]
    ].isna().mean().sort_values(ascending=False)
)

# Number of observations per station
coverage = df_year.groupby("station_id").size()

print("\nStation coverage:")
print(coverage.describe())

# Check how many stations have a full year of hourly data
expected_hours = 365 * 24  # 2003 was not a leap year

print(
    "\nStations with fewer than 8760 observations:",
    (coverage < expected_hours).sum()
)

print(
    "Stations with 8760 observations:",
    (coverage == expected_hours).sum()
)


Missingness in important variables:
era5_wbgt     0.060535
modis_ndvi    0.004018
wbgt          0.000157
dtype: float64

Station coverage:
count      66.000000
mean     8689.166667
std       249.286279
min      6802.000000
25%      8719.000000
50%      8753.000000
75%      8758.750000
max      8760.000000
dtype: float64

Stations with fewer than 8760 observations: 56
Stations with 8760 observations: 10


In [40]:
# ============================================================
# 3. Prepare annual modeling data
# ============================================================

# Require observed WBGT, ERA5 WBGT, and station ID
df_year = df_year.dropna(
    subset=["wbgt", "era5_wbgt", "station_id"]
).copy()

# Check predictor columns
missing_cols = [
    col for col in features_all
    if col not in df_year.columns
]

if missing_cols:
    raise ValueError(
        f"Missing predictor columns: {missing_cols}"
    )

# Convert predictors to numeric
df_year[features_all] = df_year[features_all].apply(
    pd.to_numeric,
    errors="coerce"
)

# Convert baseline and target to numeric
df_year["era5_wbgt"] = pd.to_numeric(
    df_year["era5_wbgt"], errors="coerce"
)

df_year["wbgt"] = pd.to_numeric(
    df_year["wbgt"], errors="coerce"
)

# Calculate residual target
df_year["wbgt_residual"] = (
    df_year["wbgt"] - df_year["era5_wbgt"]
)

print("Annual modeling observations:", len(df_year))
print("Annual modeling stations:", df_year["station_id"].nunique())

Annual modeling observations: 538679
Annual modeling stations: 62


Use 80-20 train-test split by CIMIS stations

In [41]:
# ============================================================
# 4. Train/test split by individual CIMIS stations
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df_year,
        groups=df_year["station_id"]
    )
)

train = df_year.iloc[train_idx].copy()
test = df_year.iloc[test_idx].copy()

# ============================================================
# 5. Verify station separation
# ============================================================

train_stations = set(train["station_id"].unique())
test_stations = set(test["station_id"].unique())

print("Training observations:", len(train))
print("Testing observations:", len(test))

print("Training stations:", len(train_stations))
print("Testing stations:", len(test_stations))

print(
    "Overlapping stations:",
    train_stations.intersection(test_stations)
)

print(
    "Training station proportion:",
    len(train_stations) / df_year["station_id"].nunique()
)

print(
    "Testing station proportion:",
    len(test_stations) / df_year["station_id"].nunique()
)

Training observations: 425149
Testing observations: 113530
Training stations: 49
Testing stations: 13
Overlapping stations: set()
Training station proportion: 0.7903225806451613
Testing station proportion: 0.20967741935483872


In [42]:
# ============================================================
# 6. Prepare X and y
# ============================================================

# Predictors (exclude observed WBGT and ERA5 WBGT)
X_train = train[features_all].replace(
    [np.inf, -np.inf], np.nan
)

X_test = test[features_all].replace(
    [np.inf, -np.inf], np.nan
)

# Residual target
y_train = train["wbgt_residual"].to_numpy()
y_test = test["wbgt_residual"].to_numpy()

# Preserve ERA5 baseline for reconstruction
era5_train = train["era5_wbgt"].to_numpy()
era5_test = test["era5_wbgt"].to_numpy()

# Actual CIMIS WBGT
wbgt_train = train["wbgt"].to_numpy()
wbgt_test = test["wbgt"].to_numpy()

Use the parameters found from the randomized hyperparameter search on one month model.

Mean CV RMSE: 2.2016 °C

SD across folds: 0.1934 °C

-  rf__n_estimators: 100
-  rf__min_samples_split: 5
-  rf__min_samples_leaf: 2
-  rf__max_samples: 0.7
-  rf__max_features: 0.5
-  rf__max_depth: 25

In [43]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(
        n_estimators=100,
        min_samples_split=5,
        min_samples_leaf=2,
        max_samples=0.7,
        max_features=0.5,
        max_depth=25,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

ERROR! Session/line number was not unique in database. History logging moved to new session 614


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a featu

In [44]:
# predict residuals and reconstruct CIMIS WBGT:

predicted_residual = rf_pipeline.predict(X_test)
predicted_wbgt = era5_test + predicted_residual

10-fold cv using 2003 data

In [45]:
# Model 1: All predictors, excluding the baseline WBGT
features_all = [
    col for col in features
    if col != baseline
]

# Model 2: Selected 17 predictors, excluding ERA5 WBGT
features_selected = [
    "Tair",
    "solar",
    "cza",
    "daymet_tmin",
    "gridmet_tmmn",
    "fdir",
    "relhum",
    "aster_elev_min",
    "aster_elev_q25",
    "aster_elev_mean",
    "aster_elev_median",
    "daymet_tmax",
    "aster_elev_q75",
    "elev_m",
    "aster_elev_max",
    "pres",
    "gridmet_tmmx",
]

# Model 3: Top 6 predictors, excluding ERA5 WBGT
features_top6 = [
    "Tair",
    "solar",
    "cza",
    "daymet_tmin",
    "gridmet_tmmn",
    "fdir",
]

In [46]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# ============================================================
# 1. Define target, baseline, and predictor sets
# ============================================================

target = "wbgt"
baseline = "era5_wbgt"
residual_target = "wbgt_residual"

models = {
    "RF - All predictors": features_all,
    "RF - Selected predictors": features_selected,
    "RF - Top predictors": features_top6,
}

# Check that target and baseline are excluded
for model_name, feature_cols in models.items():

    if target in feature_cols:
        raise ValueError(
            f"{model_name} includes observed WBGT as a predictor."
        )

    if baseline in feature_cols:
        raise ValueError(
            f"{model_name} includes ERA5 WBGT as a predictor."
        )

# Check predictor columns exist
for model_name, feature_cols in models.items():

    missing = set(feature_cols) - set(train.columns)

    if missing:
        raise ValueError(
            f"{model_name}: missing columns {missing}"
        )

In [47]:
# ============================================================
# 2. Prepare annual training data
# ============================================================

cv_data = train.dropna(
    subset=[target, baseline, "station_id"]
).copy()

# Calculate residuals
cv_data[residual_target] = (
    cv_data[target] - cv_data[baseline]
)

# Residual target
y = cv_data[residual_target].to_numpy()

# ERA5 baseline
era5_values = cv_data[baseline].to_numpy()

# Actual CIMIS WBGT
wbgt_values = cv_data[target].to_numpy()

# Station groups
groups = cv_data["station_id"].to_numpy()

n_stations = cv_data["station_id"].nunique()

print("Year:", cv_data["year"].unique())
print("Training observations:", len(cv_data))
print("Training stations:", n_stations)
print("Training months:", sorted(cv_data["month"].unique()))

if n_stations < 10:
    raise ValueError(
        f"Only {n_stations} stations available. "
        "10-fold GroupKFold requires at least 10 stations."
    )

Year: [2003]
Training observations: 425149
Training stations: 49
Training months: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


## one year RF 10-fold

In [48]:
# ============================================================
# 3. Set up 10-fold GroupKFold
# ============================================================

cv = GroupKFold(n_splits=10)

In [49]:
# ============================================================
# 4. Run 10-fold CV for all three models
# ============================================================

cv_results = []
start_all = time.time()

for model_name, feature_cols in models.items():

    print(f"\nRunning: {model_name}")

    X = cv_data[feature_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X, y, groups=groups), start=1
    ):

        # Split predictors
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        # Split residual targets
        y_train = y[train_idx]
        y_val = y[val_idx]

        # Validation ERA5 baseline and observed WBGT
        era5_val = era5_values[val_idx]
        wbgt_val = wbgt_values[val_idx]

        # Confirm no station overlap
        train_stations = set(groups[train_idx])
        val_stations = set(groups[val_idx])

        assert train_stations.isdisjoint(val_stations)

        # Impute missing predictors using training data only
        imputer = SimpleImputer(strategy="median")

        X_train_imp = imputer.fit_transform(X_train)
        X_val_imp = imputer.transform(X_val)

        # Random Forest using Rank 1 parameters
        rf = RandomForestRegressor(
            n_estimators=100,
            min_samples_split=5,
            min_samples_leaf=2,
            max_samples=0.7,
            max_features=0.5,
            max_depth=25,
            random_state=42,
            n_jobs=-1,
        )

        rf.fit(X_train_imp, y_train)

        # Predict residuals
        predicted_residual = rf.predict(X_val_imp)

        # Reconstruct CIMIS WBGT
        predicted_wbgt = era5_val + predicted_residual

        # ----------------------------------------------------
        # RF-corrected WBGT metrics
        # ----------------------------------------------------

        rmse = np.sqrt(
            mean_squared_error(wbgt_val, predicted_wbgt)
        )

        mae = mean_absolute_error(
            wbgt_val, predicted_wbgt
        )

        bias = np.mean(predicted_wbgt - wbgt_val)

        r2 = r2_score(wbgt_val, predicted_wbgt)

        # ----------------------------------------------------
        # Uncorrected ERA5 baseline metrics
        # ----------------------------------------------------

        era5_rmse = np.sqrt(
            mean_squared_error(wbgt_val, era5_val)
        )

        era5_mae = mean_absolute_error(
            wbgt_val, era5_val
        )

        era5_bias = np.mean(era5_val - wbgt_val)

        era5_r2 = r2_score(wbgt_val, era5_val)

        # ----------------------------------------------------
        # Residual prediction metrics
        # ----------------------------------------------------

        residual_rmse = np.sqrt(
            mean_squared_error(y_val, predicted_residual)
        )

        residual_mae = mean_absolute_error(
            y_val, predicted_residual
        )

        residual_bias = np.mean(
            predicted_residual - y_val
        )

        residual_r2 = r2_score(y_val, predicted_residual)

        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        cv_results.append({
            "Model": model_name,
            "Fold": fold,
            "Train_rows": len(train_idx),
            "Validation_rows": len(val_idx),
            "Train_stations": len(train_stations),
            "Validation_stations": len(val_stations),

            # RF-corrected WBGT
            "WBGT_RMSE": rmse,
            "WBGT_MAE": mae,
            "WBGT_Bias": bias,
            "WBGT_R2": r2,

            # ERA5 baseline
            "ERA5_RMSE": era5_rmse,
            "ERA5_MAE": era5_mae,
            "ERA5_Bias": era5_bias,
            "ERA5_R2": era5_r2,

            # Residual prediction
            "Residual_RMSE": residual_rmse,
            "Residual_MAE": residual_mae,
            "Residual_Bias": residual_bias,
            "Residual_R2": residual_r2,
        })

        print(
            f"Fold {fold:02d}/10 | "
            f"RF RMSE={rmse:.4f} | "
            f"ERA5 RMSE={era5_rmse:.4f} | "
            f"RF R2={r2:.4f}"
        )

print(
    f"\nTotal CV time: "
    f"{(time.time() - start_all) / 60:.2f} minutes"
)



Running: RF - All predictors
Fold 01/10 | RF RMSE=1.4374 | ERA5 RMSE=2.2930 | RF R2=0.9582
Fold 02/10 | RF RMSE=1.8529 | ERA5 RMSE=2.9117 | RF R2=0.9505
Fold 03/10 | RF RMSE=1.6268 | ERA5 RMSE=2.6224 | RF R2=0.9349
Fold 04/10 | RF RMSE=1.7622 | ERA5 RMSE=2.6325 | RF R2=0.9282
Fold 05/10 | RF RMSE=1.5383 | ERA5 RMSE=2.6200 | RF R2=0.9501
Fold 06/10 | RF RMSE=1.6134 | ERA5 RMSE=2.3994 | RF R2=0.9394
Fold 07/10 | RF RMSE=1.9167 | ERA5 RMSE=2.5192 | RF R2=0.9347
Fold 08/10 | RF RMSE=1.5817 | ERA5 RMSE=2.4708 | RF R2=0.9606
Fold 09/10 | RF RMSE=1.8571 | ERA5 RMSE=2.8116 | RF R2=0.9515
Fold 10/10 | RF RMSE=1.5834 | ERA5 RMSE=2.5466 | RF R2=0.9618

Running: RF - Selected predictors
Fold 01/10 | RF RMSE=1.5761 | ERA5 RMSE=2.2930 | RF R2=0.9497
Fold 02/10 | RF RMSE=1.9049 | ERA5 RMSE=2.9117 | RF R2=0.9477
Fold 03/10 | RF RMSE=1.6774 | ERA5 RMSE=2.6224 | RF R2=0.9308
Fold 04/10 | RF RMSE=1.7884 | ERA5 RMSE=2.6325 | RF R2=0.9260
Fold 05/10 | RF RMSE=1.8698 | ERA5 RMSE=2.6200 | RF R2=0.9263
Fold 

In [50]:
# ============================================================
# 5. Results for individual folds
# ============================================================

rf_results_df = pd.DataFrame(cv_results)

print("\nResults for each fold:")
print(rf_results_df.round(4).to_string(index=False))


# ============================================================
# 6. Summary across folds
# ============================================================

metrics = [
    "WBGT_RMSE",
    "WBGT_MAE",
    "WBGT_Bias",
    "WBGT_R2",
    "ERA5_RMSE",
    "ERA5_MAE",
    "ERA5_Bias",
    "ERA5_R2",
    "Residual_RMSE",
    "Residual_MAE",
    "Residual_Bias",
    "Residual_R2",
]

summary = (
    rf_results_df
    .groupby("Model")[metrics]
    .agg(["mean", "std"])
)

print("\n10-fold CV summary:")
print(summary.round(4))


Results for each fold:
                   Model  Fold  Train_rows  Validation_rows  Train_stations  Validation_stations  WBGT_RMSE  WBGT_MAE  WBGT_Bias  WBGT_R2  ERA5_RMSE  ERA5_MAE  ERA5_Bias  ERA5_R2  Residual_RMSE  Residual_MAE  Residual_Bias  Residual_R2
     RF - All predictors     1      381490            43659              44                    5     1.4374    1.1026    -0.3810   0.9582     2.2930    1.7424    -0.1624   0.8935         1.4374        1.1026        -0.3810       0.6051
     RF - All predictors     2      381704            43445              44                    5     1.8529    1.3988     0.4942   0.9505     2.9117    2.2645     0.6608   0.8779         1.8529        1.3988         0.4942       0.5730
     RF - All predictors     3      381719            43430              44                    5     1.6268    1.2213     0.0609   0.9349     2.6224    2.0289     0.4782   0.8308         1.6268        1.2213         0.0609       0.6019
     RF - All predictors     4  

In [51]:
# ============================================================
# 6. Save results
# ============================================================

rf_results_df.to_csv(
    "wbgt_rf_2003_10fold_cv_each_fold.csv",
    index=False,
)

summary.to_csv(
    "wbgt_rf_2003_10fold_cv_summary.csv"
)

print("\nSaved:")
print("- wbgt_rf_2003_10fold_cv_each_fold.csv")
print("- wbgt_rf_2003_10fold_cv_summary.csv")


Saved:
- wbgt_rf_2003_10fold_cv_each_fold.csv
- wbgt_rf_2003_10fold_cv_summary.csv


## LightGBM 10-fold cv

In [1]:
pip install lightgbm

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 11.7 MB/s  0:00:00

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import lightgbm as lgb
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [21]:
models = {
    "LightGBM - All predictors": features_all,
    "LightGBM - Selected predictors": features_selected,
    "LightGBM - Top predictors": features_top6
}

# Prepare annual training data
cv_data = train.dropna(
    subset=["wbgt", "era5_wbgt", "station_id"]
).copy()

# Residual target
cv_data["wbgt_residual"] = (
    cv_data["wbgt"] - cv_data["era5_wbgt"]
)

# Confirm predictor lists do not include the target or ERA5 baseline
for model_name, feature_cols in models.items():
    assert "wbgt" not in feature_cols
    assert "era5_wbgt" not in feature_cols

    missing_cols = set(feature_cols) - set(cv_data.columns)
    if missing_cols:
        raise ValueError(
            f"{model_name} is missing columns: {missing_cols}"
        )

# Response and baseline values
y = cv_data["wbgt_residual"].to_numpy()
wbgt_values = cv_data["wbgt"].to_numpy()
era5_values = cv_data["era5_wbgt"].to_numpy()
groups = cv_data["station_id"].to_numpy()

# Same 10-fold station-grouped CV
cv = GroupKFold(n_splits=10)

print("Observations:", len(cv_data))
print("Stations:", cv_data["station_id"].nunique())

Observations: 425149
Stations: 49


In [23]:
# parameters
lgbm_params = {
    "objective": "regression",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

In [24]:
import time

results = []

for model_name, feature_cols in models.items():

    print(f"\nRunning {model_name}...")

    # Predictor matrix
    X = (
        cv_data[feature_cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X, y, groups), start=1
    ):

        # Split data
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y[train_idx]

        wbgt_val = wbgt_values[val_idx]
        era5_val = era5_values[val_idx]
        y_val = y[val_idx]

        train_stations = np.unique(groups[train_idx])
        val_stations = np.unique(groups[val_idx])

        # Initialize model
        model = LGBMRegressor(**lgbm_params)

        # Train and time model
        start_time = time.perf_counter()

        model.fit(X_train, y_train)

        training_time = time.perf_counter() - start_time

        # Predict residuals
        predicted_residual = model.predict(X_val)

        # Reconstruct WBGT
        predicted_wbgt = era5_val + predicted_residual

        # Evaluate corrected WBGT
        wbgt_rmse = np.sqrt(
            mean_squared_error(wbgt_val, predicted_wbgt)
        )

        wbgt_mae = mean_absolute_error(
            wbgt_val, predicted_wbgt
        )

        wbgt_bias = np.mean(predicted_wbgt - wbgt_val)

        wbgt_r2 = r2_score(
            wbgt_val, predicted_wbgt
        )

        # Evaluate original ERA5 WBGT
        era5_rmse = np.sqrt(
            mean_squared_error(wbgt_val, era5_val)
        )

        era5_mae = mean_absolute_error(
            wbgt_val, era5_val
        )

        era5_bias = np.mean(era5_val - wbgt_val)

        era5_r2 = r2_score(
            wbgt_val, era5_val
        )

        # Evaluate residual predictions
        residual_rmse = np.sqrt(
            mean_squared_error(y_val, predicted_residual)
        )

        residual_mae = mean_absolute_error(
            y_val, predicted_residual
        )

        residual_bias = np.mean(
            predicted_residual - y_val
        )

        residual_r2 = r2_score(
            y_val, predicted_residual
        )

        # Store results
        results.append({
            "Model": model_name,
            "Fold": fold,
            "Train_rows": len(train_idx),
            "Validation_rows": len(val_idx),
            "Train_stations": len(train_stations),
            "Validation_stations": len(val_stations),

            "Training_time_sec": training_time,

            "WBGT_RMSE": wbgt_rmse,
            "WBGT_MAE": wbgt_mae,
            "WBGT_Bias": wbgt_bias,
            "WBGT_R2": wbgt_r2,

            "ERA5_RMSE": era5_rmse,
            "ERA5_MAE": era5_mae,
            "ERA5_Bias": era5_bias,
            "ERA5_R2": era5_r2,

            "Residual_RMSE": residual_rmse,
            "Residual_MAE": residual_mae,
            "Residual_Bias": residual_bias,
            "Residual_R2": residual_r2
        })

        print(
            f"Fold {fold:2d} | "
            f"RMSE: {wbgt_rmse:.4f} °C | "
            f"R²: {wbgt_r2:.4f} | "
            f"Time: {training_time:.2f} sec"
        )

# Convert results to DataFrame
results_df = pd.DataFrame(results)

print("\nResults for each fold:")
display(results_df)


Running LightGBM - All predictors...
Fold  1 | RMSE: 1.4760 °C | R²: 0.9559 | Time: 7.01 sec
Fold  2 | RMSE: 1.7798 °C | R²: 0.9544 | Time: 6.56 sec
Fold  3 | RMSE: 1.5433 °C | R²: 0.9414 | Time: 6.38 sec
Fold  4 | RMSE: 1.7280 °C | R²: 0.9309 | Time: 6.40 sec
Fold  5 | RMSE: 1.6135 °C | R²: 0.9451 | Time: 5.93 sec
Fold  6 | RMSE: 1.4986 °C | R²: 0.9477 | Time: 7.65 sec
Fold  7 | RMSE: 2.0243 °C | R²: 0.9272 | Time: 8.16 sec
Fold  8 | RMSE: 1.4803 °C | R²: 0.9655 | Time: 10.58 sec
Fold  9 | RMSE: 1.8160 °C | R²: 0.9536 | Time: 11.25 sec
Fold 10 | RMSE: 1.5552 °C | R²: 0.9631 | Time: 8.07 sec

Running LightGBM - Selected predictors...
Fold  1 | RMSE: 1.5662 °C | R²: 0.9503 | Time: 5.52 sec
Fold  2 | RMSE: 1.9279 °C | R²: 0.9465 | Time: 4.95 sec
Fold  3 | RMSE: 1.6989 °C | R²: 0.9290 | Time: 5.14 sec
Fold  4 | RMSE: 1.7914 °C | R²: 0.9258 | Time: 4.52 sec
Fold  5 | RMSE: 1.8489 °C | R²: 0.9279 | Time: 5.11 sec
Fold  6 | RMSE: 1.9783 °C | R²: 0.9089 | Time: 5.23 sec
Fold  7 | RMSE: 2.110

,Model,Fold,Train_rows,Validation_rows,Train_stations,Validation_stations,Training_time_sec,WBGT_RMSE,WBGT_MAE,WBGT_Bias,WBGT_R2,ERA5_RMSE,ERA5_MAE,ERA5_Bias,ERA5_R2,Residual_RMSE,Residual_MAE,Residual_Bias,Residual_R2
0,LightGBM - All predictors,1,381490,43659,44,5,7.009374,1.475998,1.150394,-0.435744,0.955875,2.293001,1.742366,-0.162367,0.893506,1.475998,1.150394,-0.435744,0.583566
1,LightGBM - All predictors,2,381704,43445,44,5,6.555964,1.779768,1.353573,0.473802,0.954367,2.911676,2.264474,0.660788,0.877866,1.779768,1.353573,0.473802,0.606083
2,LightGBM - All predictors,3,381719,43430,44,5,6.379910,1.543290,1.175847,0.053519,0.941403,2.622359,2.028904,0.478158,0.830813,1.543290,1.175847,0.053519,0.641742
3,LightGBM - All predictors,4,381740,43409,44,5,6.398772,1.728017,1.335652,0.311176,0.930915,2.632482,2.042015,0.720120,0.839668,1.728017,1.335652,0.311176,0.534260
4,LightGBM - All predictors,5,381606,43543,44,5,5.929015,1.613520,1.247051,0.558853,0.945096,2.620029,2.073346,0.928242,0.855234,1.613520,1.247051,0.558853,0.566303
5,LightGBM - All predictors,6,381497,43652,44,5,7.645237,1.498576,1.163285,0.065904,0.947738,2.399424,1.805408,0.430492,0.866020,1.498576,1.163285,0.065904,0.596955
6,LightGBM - All predictors,7,381587,43562,44,5,8.159877,2.024290,1.542929,-1.022884,0.927212,2.519230,1.912757,-0.257681,0.887268,2.024290,1.542929,-1.022884,0.347504
7,LightGBM - All predictors,8,381494,43655,44,5,10.583125,1.480338,1.126445,-0.148477,0.965471,2.470765,1.928253,0.470700,0.903812,1.480338,1.126445,-0.148477,0.627510
8,LightGBM - All predictors,9,383351,41798,44,5,11.250105,1.816050,1.442987,-0.317310,0.953644,2.811563,2.232397,-0.045837,0.888891,1.816050,1.442987,-0.317310,0.582674
9,LightGBM - All predictors,10,390153,34996,45,4,8.070969,1.555194,1.228026,0.062284,0.963111,2.546635,1.966414,0.392290,0.901086,1.555194,1.228026,0.062284,0.617998


In [25]:
metrics = [
    "WBGT_RMSE",
    "WBGT_MAE",
    "WBGT_Bias",
    "WBGT_R2",
    "ERA5_RMSE",
    "ERA5_MAE",
    "ERA5_Bias",
    "ERA5_R2",
    "Residual_RMSE",
    "Residual_MAE",
    "Residual_Bias",
    "Residual_R2",
    "Training_time_sec"
]

summary = (
    results_df
    .groupby("Model")[metrics]
    .agg(["mean", "std"])
)

print("\n10-fold CV summary:")
display(summary.round(4))


10-fold CV summary:


WBGT_RMSE         WBGT_MAE         WBGT_Bias  \
                                    mean     std     mean     std      mean   
Model                                                                         
LightGBM - All predictors         1.6515  0.1808   1.2766  0.1385   -0.0399   
LightGBM - Selected predictors    1.8438  0.1877   1.4407  0.1628    0.0230   
LightGBM - Top predictors         1.8714  0.1432   1.4528  0.1135   -0.0018   

                                       WBGT_R2         ERA5_RMSE         ...  \
                                   std    mean     std      mean    std  ...   
Model                                                                    ...   
LightGBM - All predictors       0.4700  0.9485  0.0127    2.5827  0.183  ...   
LightGBM - Selected predictors  0.4553  0.9357  0.0158    2.5827  0.183  ...   
LightGBM - Top predictors       0.3302  0.9346  0.0106    2.5827  0.183  ...   

                               Residual_RMSE         Residual_MAE          \
                                        mean     std         mean     std   
Model                                                                       
LightGBM - All predictors             1.6515  0.1808       1.2766  0.1385   
LightGBM - Selected predictors        1.8438  0.1877       1.4407  0.1628   
LightGBM - Top predictors             1.8714  0.1432       1.4528  0.1135   

                               Residual_Bias         Residual_R2          \
                                        mean     std        mean     std   
Model                                                                      
LightGBM - All predictors            -0.0399  0.4700      0.5705  0.0843   
LightGBM - Selected predictors        0.0230  0.4553      0.4638  0.1006   
LightGBM - Top predictors            -0.0018  0.3302      0.4528  0.0292   

                               Training_time_sec          
                                            mean     std  
Model                                                     
LightGBM - All predictors                 7.7982  1.8114  
LightGBM - Selected predictors            5.2057  0.3074  
LightGBM - Top predictors                 3.3570  0.3627  

[3 rows x 26 columns]

In [26]:
results_df.to_csv(
    "lightgbm_2003_cv_fold_results.csv",
    index=False
)

summary.to_csv(
    "lightgbm_2003_cv_summary.csv"
)

## compare RF and LightGBM

In [52]:
# Combine fold-level results
combined_results = pd.concat(
    [rf_results_df, results_df],
    ignore_index=True
)

# Summarize both algorithms
comparison = (
    combined_results
    .groupby("Model")[
        [
            "WBGT_RMSE",
            "WBGT_MAE",
            "WBGT_Bias",
            "WBGT_R2",
            "Training_time_sec"
        ]
    ]
    .agg(["mean", "std"])
)

display(comparison.round(4))

WBGT_RMSE         WBGT_MAE         WBGT_Bias  \
                                    mean     std     mean     std      mean   
Model                                                                         
LightGBM - All predictors         1.6515  0.1808   1.2766  0.1385   -0.0399   
LightGBM - Selected predictors    1.8438  0.1877   1.4407  0.1628    0.0230   
LightGBM - Top predictors         1.8714  0.1432   1.4528  0.1135   -0.0018   
RF - All predictors               1.6770  0.1596   1.2881  0.1277   -0.0070   
RF - Selected predictors          1.8204  0.1714   1.4074  0.1474    0.0154   
RF - Top predictors               1.8767  0.1422   1.4561  0.1110   -0.0289   

                                       WBGT_R2         Training_time_sec  \
                                   std    mean     std              mean   
Model                                                                      
LightGBM - All predictors       0.4700  0.9485  0.0127            7.7982   
LightGBM - Selected predictors  0.4553  0.9357  0.0158            5.2057   
LightGBM - Top predictors       0.3302  0.9346  0.0106            3.3570   
RF - All predictors             0.4236  0.9470  0.0119               NaN   
RF - Selected predictors        0.3900  0.9372  0.0153               NaN   
RF - Top predictors             0.3244  0.9342  0.0105               NaN   

                                        
                                   std  
Model                                   
LightGBM - All predictors       1.8114  
LightGBM - Selected predictors  0.3074  
LightGBM - Top predictors       0.3627  
RF - All predictors                NaN  
RF - Selected predictors           NaN  
RF - Top predictors                NaN

In [53]:
# Extract all-predictor results
lgb_all = results_df[
    results_df["Model"] == "LightGBM - All predictors"
].sort_values("Fold")

rf_all = rf_results_df[
    rf_results_df["Model"] == "RF - All predictors"
].sort_values("Fold")

comparison = pd.DataFrame({
    "Fold": lgb_all["Fold"].values,
    "LightGBM_RMSE": lgb_all["WBGT_RMSE"].values,
    "RF_RMSE": rf_all["WBGT_RMSE"].values
})

comparison["RMSE_difference"] = (
    comparison["RF_RMSE"] - comparison["LightGBM_RMSE"]
)

display(comparison.round(4))

print(
    "Folds where LightGBM performed better:",
    (comparison["RMSE_difference"] > 0).sum()
)

,Fold,LightGBM_RMSE,RF_RMSE,RMSE_difference
0,1,1.4760,1.4374,-0.0386
1,2,1.7798,1.8529,0.0731
2,3,1.5433,1.6268,0.0835
3,4,1.7280,1.7622,0.0342
4,5,1.6135,1.5383,-0.0752
5,6,1.4986,1.6134,0.1148
6,7,2.0243,1.9167,-0.1076
7,8,1.4803,1.5817,0.1013
8,9,1.8160,1.8571,0.0410
9,10,1.5552,1.5834,0.0282


Folds where LightGBM performed better: 7


Overall, LightGBM performs much faster and slightly more accurate than random forest.